In [9]:

# 3. Import all necessary libraries
import pandas as pd
import numpy as np
import ast
import re
import pickle
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, GlobalMaxPooling1D, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.initializers import Constant
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


print("Environment setup complete. GloVe vectors downloaded and extracted.")

Environment setup complete. GloVe vectors downloaded and extracted.


In [10]:
# 1. Load and reshape the dataset
df = pd.read_csv('../data/master_clauses.csv')
clause_cols = [col for col in df.columns if not col.endswith('-Answer') and col != 'Filename']
df_long = df[clause_cols].melt(var_name='Clause_Type', value_name='Raw_Text')

# 2. Define the text cleaner
def clean_extracted_text(text):
    if pd.isna(text) or text == '[]':
        return None
    try:
        if isinstance(text, str) and text.startswith('['):
            extracted_list = ast.literal_eval(text)
            if isinstance(extracted_list, list) and len(extracted_list) > 0:
                text = " ".join(extracted_list)
    except:
        pass
    text = str(text).lower()
    text = re.sub(r'[\n\t\r]', ' ', text)
    text = re.sub(r'[^a-z0-9\s.,;]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 10 else None

# 3. Apply the cleaner
df_long['Cleaned_Text'] = df_long['Raw_Text'].apply(clean_extracted_text)
df_clean = df_long.dropna(subset=['Cleaned_Text']).copy()

print(f"Data Engineering complete. Total clean examples: {len(df_clean)}")

Data Engineering complete. Total clean examples: 6645


In [11]:
# 1. Hyperparameters
VOCAB_SIZE = 12000
MAX_SEQUENCE_LENGTH = 250

# 2. Encode Labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df_clean['Clause_Type'])
num_classes = len(label_encoder.classes_)

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df_clean['Cleaned_Text'], y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# 4. Tokenize Text
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQUENCE_LENGTH, padding='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_SEQUENCE_LENGTH, padding='post')

print(f"Vocabulary built. Training sequences shape: {X_train_seq.shape}")

Vocabulary built. Training sequences shape: (5316, 250)


In [12]:
print("Indexing Stanford GloVe word vectors...")
embeddings_index = {}

# 1. Load the 100D vectors into memory
with open('../glove/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

# 2. Build the exact matrix for your model
EMBEDDING_DIM = 100
word_index = tokenizer.word_index
num_tokens = min(VOCAB_SIZE, len(word_index) + 1)
embedding_matrix = np.zeros((num_tokens, EMBEDDING_DIM))

hits = 0
misses = 0

for word, i in word_index.items():
    if i >= VOCAB_SIZE:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1

print(f"Matrix built! Matched {hits} words. Missed {misses} words.")

Indexing Stanford GloVe word vectors...
Matrix built! Matched 7958 words. Missed 2009 words.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the visual style for the dashboard
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Lex-Assist: Deep Learning & Data Analytics Dashboard', fontsize=16, fontweight='bold')



In [ ]:
# --- Chart 1: Clause Distribution (Pandas & Seaborn) ---
# Count how many of each clause type exist in the clean data
clause_counts = df_clean['Clause_Type'].value_counts()
sns.barplot(x=clause_counts.values, y=clause_counts.index, ax=axes[0], palette="viridis")
axes[0].set_title('Distribution of Legal Clauses')
axes[0].set_xlabel('Number of Samples')
axes[0].set_ylabel('Clause Type')


In [ ]:
# --- Chart 2: Sequence Lengths (Statistical Analysis) ---
# Calculate the word count for each clean text
df_clean['Word_Count'] = df_clean['Cleaned_Text'].apply(lambda x: len(x.split()))
sns.histplot(df_clean['Word_Count'], bins=50, kde=True, ax=axes[1], color="coral")
axes[1].axvline(MAX_SEQUENCE_LENGTH, color='red', linestyle='--', label=f'Max Length ({MAX_SEQUENCE_LENGTH})')
axes[1].set_title('Word Count Distribution per Clause')
axes[1].set_xlabel('Number of Words')
axes[1].set_ylabel('Frequency')
axes[1].legend()


In [13]:
# 1. Build the GloVe-Powered Architecture
model = Sequential([
    Embedding(
        input_dim=num_tokens,
        output_dim=EMBEDDING_DIM,
        embeddings_initializer=Constant(embedding_matrix),
        input_length=MAX_SEQUENCE_LENGTH,
        trainable=False  # Freeze the pre-trained weights
    ),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dropout(0.4),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# 2. Define Callbacks to optimize training
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2)
]

print("🚀 Commencing Optimized Deep Learning Training...")

# 3. Train the Model
history = model.fit(
    X_train_seq,
    y_train,
    epochs=30,
    validation_data=(X_test_seq, y_test),
    batch_size=64,
    callbacks=callbacks
)

d:\Lex_Assist_AI\.venv\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


🚀 Commencing Optimized Deep Learning Training...
Epoch 1/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 28s 289ms/step - accuracy: 0.2459 - loss: 3.2174 - val_accuracy: 0.3815 - val_loss: 2.5528 - learning_rate: 0.0010
Epoch 2/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 24s 286ms/step - accuracy: 0.4074 - loss: 2.3604 - val_accuracy: 0.5019 - val_loss: 1.9555 - learning_rate: 0.0010
Epoch 3/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 23s 267ms/step - accuracy: 0.4769 - loss: 1.9521 - val_accuracy: 0.5267 - val_loss: 1.6737 - learning_rate: 0.0010
Epoch 4/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 23s 272ms/step - accuracy: 0.5265 - loss: 1.7160 - val_accuracy: 0.5756 - val_loss: 1.5005 - learning_rate: 0.0010
Epoch 5/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 20s 235ms/step - accuracy: 0.5636 - loss: 1.5442 - val_accuracy: 0.5974 - val_loss: 1.3632 - learning_rate: 0.0010
Epoch 6/30
84/84 ━━━━━━━━━━━━━━━━━━━━ 19s 222ms/step - accuracy: 0.5961 - loss: 1.4220 - val_accuracy: 0.6208 - val_loss: 1.2701 - learning_rate: 0.0010
Epoch 7/30
84/84 ━━━━━━━━━━━━━━━━

In [ ]:
# --- Chart 3: Model Training History (Evaluation) ---
# Plot training vs validation accuracy
axes[2].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[2].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, linestyle='--')
axes[2].set_title('BiLSTM Model Accuracy over Epochs')
axes[2].set_xlabel('Epochs')
axes[2].set_ylabel('Accuracy')
axes[2].legend(loc='lower right')

plt.tight_layout()
plt.show()

# Optional: Save it as an image for your GitHub README!
fig.savefig("lex_assist_analytics.png", dpi=300)

In [15]:
# 1. Package all deep learning assets into a single dictionary
dl_pipeline_artifacts = {
    "tokenizer": tokenizer,
    "label_encoder": label_encoder,
    "model_config": model.to_json(),
    "model_weights": model.get_weights()
}

# 2. Save as a pickle file
pickle_filename = "bilstm_clause_pipeline.pkl"
with open(pickle_filename, "wb") as handle:
    pickle.dump(dl_pipeline_artifacts, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("✅ Pipeline successfully packed!")

# 3. Trigger download to your local computer


✅ Pipeline successfully packed!
